In [28]:
import requests
import pandas as pd
from bs4 import BeautifulSoup as bs
from urllib.parse import urljoin
import re
import time

In [54]:
# my_headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.14; rv:84.0) Gecko/20100101 Firefox/84.0"}
updated_headers = {"User=Agent":"Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:160.0) Gecko/20100101 Firefox/160.0"}
url = 'https://www.goodreads.com/list/show/4902.YA_Novels_of_2012'
req = requests.get(url, headers = my_headers)
req.status_code

200

In [55]:
soup = bs(req.content)

In [56]:

books = []

for row in soup.select("tr[itemscope][itemtype='http://schema.org/Book']"):

    # Title and URL
    title_tag = row.select_one("a.bookTitle")

    if not title_tag:
        continue

    title = title_tag.get_text(strip=True)
    book_url = urljoin("https://www.goodreads.com", title_tag.get("href", ""))

    # Author
    author_tag = row.select_one('[itemprop="author"]')
    author = author_tag.get_text(strip=True) if author_tag else None

    # Rating and rating count
    rating = None
    rating_count = None

    rating_tag = row.select_one(".minirating")

    if rating_tag:
        rating_text = rating_tag.get_text(" ", strip=True)

        match = re.search(
            r"([\d.]+)\s+avg rating\s*[—–-]\s*([\d,]+)\s+ratings",
            rating_text
        )

        if match:
            rating = float(match.group(1))
            rating_count = int(match.group(2).replace(",", ""))

    # Rank
    rank_tag = row.select_one("td.number")
    rank = rank_tag.get_text(strip=True) if rank_tag else None

    # scores
    small_text = row.find("span", class_="smallText uitext")
    
    if small_text:
    
        # Extract score from the <a> link
        score_link = small_text.find("a")
    
        if score_link:
            score_match = re.search(
                r"[\d,]+",
                score_link.get_text(strip=True)
            )
    
            if score_match:
                score = int(score_match.group().replace(",", ""))

    # Number of people who voted
    votes = None

    votes_tag = row.select_one("span.smallText.uitext")

    if votes_tag:
        votes_match = re.search(
            r"([\d,]+)\s+people voted",
            votes_tag.get_text(" ", strip=True)
        )

        if votes_match:
            votes = int(votes_match.group(1).replace(",", ""))

    # Store book information
    books.append({
        "rank": rank,
        "title": title,
        "author": author,
        "rating": rating,
        "rating_count": rating_count,
        "score": score,
        "votes": votes,
        "url": book_url
    })

# # Print results
# for book in books:
#     print(book)

# print(f"\nTotal books found: {len(books)}")


In [57]:
# 13.38

df = pd.DataFrame(books)
df

,rank,title,author,rating,rating_count,score,votes,url
0,1,"Insurgent (Divergent, #2)",Veronica Roth(Goodreads Author),3.96,1664723,355860,None,https://www.goodreads.com/book/show/11735983-i...
1,2,"City of Lost Souls (The Mortal Instruments, #5)",Cassandra Clare(Goodreads Author),4.20,578959,290485,None,https://www.goodreads.com/book/show/8755776-ci...
2,3,The Fault in Our Stars,John Green(Goodreads Author),4.12,5919793,234097,None,https://www.goodreads.com/book/show/11870085-t...
3,4,"Pandemonium (Delirium, #2)",Lauren Oliver(Goodreads Author),4.02,247799,166110,None,https://www.goodreads.com/book/show/9593911-pa...
4,5,"Rapture (Fallen, #4)",Lauren Kate(Goodreads Author),3.98,127692,116595,None,https://www.goodreads.com/book/show/12716010-r...
...,...,...,...,...,...,...,...,...
95,96,"Slide (Slide, #1)",Jill Hathaway(Goodreads Author),3.69,7432,7903,None,https://www.goodreads.com/book/show/9542582-slide
96,97,Above,Leah Bobet(Goodreads Author),3.12,2876,7897,None,https://www.goodreads.com/book/show/11250671-a...
97,98,"Fracture (Fracture, #1)",Megan Miranda(Goodreads Author),3.72,15741,7802,None,https://www.goodreads.com/book/show/9548964-fr...
98,99,"Rebel Heart (Dust Lands, #2)",Moira Young(Goodreads Author),3.87,20065,7516,None,https://www.goodreads.com/book/show/13042154-r...


## API query for Westminster

> all the votes of Diane Abbott

In [29]:

BASE = "https://commonsvotes-api.parliament.uk/data/divisions.json"

def get_member_votes(member_id, start_date=None, end_date=None, search_term=None):
    """Return every Commons vote by one MP as a DataFrame."""
    params = {"queryParameters.memberId": member_id}
    if start_date:
        params["queryParameters.startDate"] = start_date   # "yyyy-MM-dd"
    if end_date:
        params["queryParameters.endDate"] = end_date
    if search_term:
        params["queryParameters.searchTerm"] = search_term

    # how many to expect (optional, but useful for progress)
    total = requests.get(f"{BASE}/searchTotalResults", params=params, timeout=30).json()
    print(f"Expecting about {total} divisions")

    rows, skip = [], 0
    while True:
        page_params = {**params,
                       "queryParameters.skip": skip,
                       "queryParameters.take": 25}      # 25 is the API's maximum
        r = requests.get(f"{BASE}/membervoting", params=page_params, timeout=30)
        r.raise_for_status()
        page = r.json()
        if not page:                                    # empty list = finished
            break

        for rec in page:
            d = rec["PublishedDivision"]
            rows.append({
                "division_id": d["DivisionId"],
                "date":        d["Date"][:10],
                "division_no": d["Number"],
                "title":       d["Title"],
                "voted":       "Aye" if rec["MemberVotedAye"] else "No",
                "teller":      rec["MemberWasTeller"],
                "ayes":        d["AyeCount"],
                "noes":        d["NoCount"],
                "url":         f"https://votes.parliament.uk/Votes/Commons/Division/{d['DivisionId']}",
            })

        skip += 25
        print(f"  {len(rows)} so far")
        time.sleep(0.2)                                 # be polite to the server

    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"])
    return df


In [30]:

df = get_member_votes(172)

print(df["voted"].value_counts())


Expecting about 1527 divisions
  25 so far
  50 so far
  75 so far
  100 so far
  125 so far
  150 so far
  175 so far
  200 so far
  225 so far
  250 so far
  275 so far
  300 so far
  325 so far
  350 so far
  375 so far
  400 so far
  425 so far
  450 so far
  475 so far
  500 so far
  525 so far
  550 so far
  575 so far
  600 so far
  625 so far
  650 so far
  675 so far
  700 so far
  725 so far
  750 so far
  775 so far
  800 so far
  825 so far
  850 so far
  875 so far
  900 so far
  925 so far
  950 so far
  975 so far
  1000 so far
  1025 so far
  1050 so far
  1075 so far
  1100 so far
  1125 so far
  1150 so far
  1175 so far
  1200 so far
  1225 so far
  1250 so far
  1275 so far
  1300 so far
  1325 so far
  1350 so far
  1375 so far
  1400 so far
  1425 so far
  1450 so far
  1475 so far
  1500 so far
  1525 so far
  1527 so far
voted
Aye    851
No     676
Name: count, dtype: int64


In [31]:
df

,division_id,date,division_no,title,voted,teller,ayes,noes,url
0,2486,2026-11-17,127,Immigration and Asylum Bill: Third Reading,No,False,290,94,https://votes.parliament.uk/Votes/Commons/Divi...
1,2485,2026-11-17,126,Immigration and Asylum Bill Report Stage: New ...,Aye,False,44,368,https://votes.parliament.uk/Votes/Commons/Divi...
2,2484,2026-11-17,125,Immigration and Asylum Bill Report Stage: New ...,No,False,99,384,https://votes.parliament.uk/Votes/Commons/Divi...
3,2483,2026-11-17,124,Immigration and Asylum Bill Report Stage: New ...,No,False,102,382,https://votes.parliament.uk/Votes/Commons/Divi...
4,2428,2026-09-11,75,Terminally Ill Adults (End of Life) Bill: Seco...,No,False,270,286,https://votes.parliament.uk/Votes/Commons/Divi...
...,...,...,...,...,...,...,...,...,...
1522,16,2016-03-14,218,Energy Bill (Lord) Report Stage: New Clause 10,Aye,False,229,275,https://votes.parliament.uk/Votes/Commons/Divi...
1523,15,2016-03-14,217,Energy Bill (Lords) Report Stage: New Clause 8,Aye,False,227,272,https://votes.parliament.uk/Votes/Commons/Divi...
1524,14,2016-03-14,216,Energy Bill (Lords) Report Stage: New Clause 3,Aye,False,229,268,https://votes.parliament.uk/Votes/Commons/Divi...
1525,13,2016-03-14,215,Energy Bill: amendment 8,Aye,False,229,271,https://votes.parliament.uk/Votes/Commons/Divi...
